# Colab-TTS-Host 开发与真实模型验收
使用免费 CPU/T4（实际可用性以分配结果为准）。这里不部署公开 server，不创建保活隧道。
模型推理测试使用上游公开参考音频；个人声音训练需要另外上传有授权的数据。

In [ ]:
import pathlib, subprocess
repo = pathlib.Path("/content/Colab-TTS-Host")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", "feat/streaming-speech-server", "https://github.com/hewenyu/Colab-TTS-Host.git", str(repo)], check=True)
%cd /content/Colab-TTS-Host

In [ ]:
import shutil, subprocess
gpu = shutil.which("nvidia-smi") is not None
mode = "gpu" if gpu else "cpu"
subprocess.run(["bash", "scripts/bootstrap_colab.sh", mode], check=True)

In [ ]:
! .venv/bin/pytest -q

In [ ]:
config = "config/gpu.yaml" if gpu else "config/server.yaml"
subprocess.run([".venv/bin/python", "scripts/smoke_models.py", "--config", config], check=True)

In [ ]:
import json
print(json.dumps(json.loads(pathlib.Path("data/smoke-report.json").read_text()), ensure_ascii=False, indent=2))

## 自己的声音
先通过 `/v1/voices` 上传样本获得零样本音色；微调见 `docs/training.md`。只有确认 CUDA GPU 已分配后才启动训练。
训练结果在 Colab 断开前运行 `scripts/upload_weights.py` 保存；HF_TOKEN 通过 Colab Secrets 读取，不写入 notebook。
开发完成将代码提交 GitHub，由目标服务器或构建机执行 Docker 构建。Colab 不需要安装 Docker daemon。